In [0]:
%pip install soda-core-spark-df

In [0]:
# =============================================================================
# F1-Pulse | Data Quality — Silver Layer
# Notebook: 03_Check_Silver
# Runs SODA checks against Silver tables.
# Raises exception on failure to halt the Databricks job.
# =============================================================================



import os
from soda.scan import Scan
from config.config import CATALOG, SILVER, MIN_LAP_DURATION_S

# ---------------------------------------------------------------------------
# 1. Load Silver tables from metastore
# ---------------------------------------------------------------------------
df_sessions = spark.table(f"{CATALOG}.{SILVER}.cleaned_sessions")
df_laps     = spark.table(f"{CATALOG}.{SILVER}.enriched_laps")

# ---------------------------------------------------------------------------
# 2. Register temp views (underscore names — dots not allowed in view names)
# ---------------------------------------------------------------------------
df_sessions.createOrReplaceTempView("silver_cleaned_sessions")
df_laps.createOrReplaceTempView("silver_enriched_laps")

# ---------------------------------------------------------------------------
# 3. Load and patch YAML
#    - Patch table names to match temp view names
#    - Patch lap duration threshold from central config (single source of truth)
# ---------------------------------------------------------------------------
repo_root = "/Workspace/Users/jafar.gahramanov@gmail.com/F1-Pulse"
with open(f"{repo_root}/soda/checks_silver.yml", "r") as f:
    yaml_content = f.read()

# Table name patches
yaml_content = yaml_content.replace(
    "checks for silver.cleaned_sessions",
    "checks for silver_cleaned_sessions"
)
yaml_content = yaml_content.replace(
    "checks for silver.enriched_laps",
    "checks for silver_enriched_laps"
)

# Threshold patch — keeps YAML in sync with config.py automatically
yaml_content = yaml_content.replace(">= 60", f">= {MIN_LAP_DURATION_S}")

# ---------------------------------------------------------------------------
# 4. Run Soda scan
# ---------------------------------------------------------------------------
scan = Scan()
scan.set_data_source_name("spark_df")
scan.add_spark_session(spark)
scan.add_sodacl_yaml_str(yaml_content)

exit_code = scan.execute()
print(scan.get_logs_text())

# ---------------------------------------------------------------------------
# 5. Halt pipeline on failure — Databricks job will mark task as FAILED
# ---------------------------------------------------------------------------
if exit_code != 0:
    raise Exception("❌ Silver DQ checks failed. Halting pipeline.")

print("✅ Silver DQ checks passed.")